In [1]:
# %%
# ============================================
# SIMPLIFIED & WORKING FISH DISEASE DETECTION
# ============================================

# 1. INSTALL REQUIRED LIBRARIES
print("📦 Installing required libraries...")
!pip install tensorflow matplotlib opencv-python scikit-learn seaborn pandas numpy pillow -q

📦 Installing required libraries...
^C



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# %%
# 2. IMPORT LIBRARIES
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import os
import zipfile
import shutil
import random
import json
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from google.colab import files

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.optimizers import Adam

print("✅ Libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")

In [ ]:
# %%
# 3. SIMPLE DATASET LOADER
def load_dataset_from_folders(dataset_path, img_size=224):
    """
    Simple dataset loader - expects folders as classes
    """
    print(f"\n📁 Loading dataset from: {dataset_path}")

    images = []
    labels = []

    # Get all folders (classes)
    class_folders = [f for f in os.listdir(dataset_path)
                    if os.path.isdir(os.path.join(dataset_path, f))]

    if not class_folders:
        print("❌ No folders found! Looking for images in root...")
        image_files = [f for f in os.listdir(dataset_path)
                      if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]

        if not image_files:
            print("❌ No images found!")
            return None, None, None, None

        class_folders = ['fish_disease']
        print(f"⚠️ Found {len(image_files)} images in root, treating as single class")

    print(f"✅ Found {len(class_folders)} classes: {class_folders}")

    label_map = {class_name: idx for idx, class_name in enumerate(sorted(class_folders))}
    reverse_label_map = {idx: class_name for class_name, idx in label_map.items()}

    total_images = 0
    for class_name in sorted(class_folders):
        class_path = os.path.join(dataset_path, class_name)

        if os.path.isdir(class_path):
            image_files = [f for f in os.listdir(class_path)
                          if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
        else:
            image_files = [f for f in os.listdir(dataset_path)
                          if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
            class_path = dataset_path

        print(f"📁 Loading {class_name}: {len(image_files)} images")

        loaded = 0
        for img_file in image_files[:500]:
            img_path = os.path.join(class_path, img_file)

            try:
                img = cv2.imread(img_path)
                if img is None:
                    continue

                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (img_size, img_size))

                images.append(img)
                labels.append(label_map[class_name])
                loaded += 1
                total_images += 1

            except Exception as e:
                print(f"  ⚠️ Error loading {img_file}: {e}")

        print(f"  ✅ Loaded {loaded} images")

    if total_images == 0:
        print("❌ No images loaded!")
        return None, None, None, None

    print(f"\n🎉 Successfully loaded {total_images} images from {len(class_folders)} classes")
    return np.array(images), np.array(labels), class_folders, reverse_label_map

In [ ]:
# %%
# 4. UPLOAD DATASET
def upload_dataset():
    """Upload and extract dataset"""
    print("\n" + "="*60)
    print("📤 UPLOAD YOUR DATASET")
    print("="*60)
    print("Please upload your ZIP file")

    uploaded = files.upload()

    zip_filename = None
    for filename in uploaded.keys():
        if filename.endswith('.zip'):
            zip_filename = filename
            file_size = len(uploaded[filename]) / (1024 * 1024)
            print(f"\n✅ Uploaded: {filename} ({file_size:.2f} MB)")
            break

    if not zip_filename:
        print("❌ No ZIP file found!")
        return None

    extract_dir = 'fish_dataset'
    if os.path.exists(extract_dir):
        shutil.rmtree(extract_dir)

    os.makedirs(extract_dir, exist_ok=True)

    print(f"\n📂 Extracting...")
    with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)

    items = os.listdir(extract_dir)
    if len(items) == 1 and os.path.isdir(os.path.join(extract_dir, items[0])):
        extract_dir = os.path.join(extract_dir, items[0])
        print(f"📁 Using inner folder: {items[0]}")

    print("✅ Extraction complete!")
    return extract_dir

In [ ]:
# %%
# 5. CREATE SIMPLE MODEL
def create_simple_model(num_classes, img_size=224):
    """Create a simple working model"""
    print(f"\n🤖 Creating model for {num_classes} classes...")

    base_model = MobileNetV2(
        input_shape=(img_size, img_size, 3),
        include_top=False,
        weights='imagenet'
    )
    base_model.trainable = False

    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.3),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ])

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    model.summary()
    return model

In [ ]:
# %%
# 6. TRAIN MODEL
def train_simple_model(model, X_train, y_train, X_val, y_val, epochs=20, batch_size=32):
    """Simple training function"""
    print("\n" + "="*60)
    print("🚀 TRAINING STARTED")
    print("="*60)

    datagen = ImageDataGenerator(
        rotation_range=15,
        width_shift_range=0.1,
        height_shift_range=0.1,
        zoom_range=0.1,
        horizontal_flip=True,
        fill_mode='nearest'
    )

    callbacks = [
        EarlyStopping(patience=8, restore_best_weights=True, verbose=1),
        ModelCheckpoint('best_model.h5', save_best_only=True, monitor='val_accuracy', verbose=1),
        ReduceLROnPlateau(factor=0.5, patience=4, verbose=1)
    ]

    steps_per_epoch = max(1, len(X_train) // batch_size)

    print(f"📊 Training Info:")
    print(f"  Images: {len(X_train)} train, {len(X_val)} validation")
    print(f"  Batch size: {batch_size}")
    print(f"  Steps per epoch: {steps_per_epoch}")
    print(f"  Epochs: {epochs}")

    print("\n⏳ Training...")
    history = model.fit(
        datagen.flow(X_train, y_train, batch_size=batch_size),
        steps_per_epoch=steps_per_epoch,
        epochs=epochs,
        validation_data=(X_val, y_val),
        callbacks=callbacks,
        verbose=1
    )

    return history

In [ ]:
# %%
# 7. EVALUATE AND VISUALIZE
def evaluate_and_visualize(model, X_test, y_test, class_names, history):
    """Evaluate model and show results"""
    print("\n" + "="*60)
    print("📊 EVALUATION RESULTS")
    print("="*60)

    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
    print(f"✅ Test Accuracy: {test_acc*100:.2f}%")
    print(f"✅ Test Loss: {test_loss:.4f}")

    y_pred = model.predict(X_test, verbose=0)
    y_pred_classes = np.argmax(y_pred, axis=1)

    print("\n📋 Classification Report:")
    print(classification_report(y_test, y_pred_classes, target_names=class_names))

    print("\n🎯 Confusion Matrix:")
    cm = confusion_matrix(y_test, y_pred_classes)

    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title('Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.show()

    # Plot training history
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Val Accuracy')
    plt.title('Model Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Val Loss')
    plt.title('Model Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

    # Show sample predictions
    print("\n🔍 Sample Predictions:")
    num_samples = min(8, len(X_test))
    sample_indices = np.random.choice(len(X_test), num_samples, replace=False)

    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()

    for idx, ax in enumerate(axes):
        if idx < len(sample_indices):
            i = sample_indices[idx]

            ax.imshow(X_test[i])

            true_class = class_names[y_test[i]]
            pred_class = class_names[y_pred_classes[i]]
            confidence = np.max(y_pred[i]) * 100

            color = 'green' if true_class == pred_class else 'red'
            ax.set_title(f"True: {true_class}\nPred: {pred_class}\nConf: {confidence:.1f}%",
                        color=color, fontsize=10)
            ax.axis('off')
        else:
            ax.axis('off')

    plt.tight_layout()
    plt.show()

    return test_acc

In [ ]:
# %%
# 8. SAVE MODEL
def save_final_model(model, class_names, reverse_label_map, test_accuracy):
    """Save the trained model"""
    print("\n" + "="*60)
    print("💾 SAVING MODEL")
    print("="*60)

    model.save('fish_disease_model_final.h5')
    print("✅ Model saved: fish_disease_model_final.h5")

    class_info = {
        'class_names': class_names,
        'reverse_label_map': reverse_label_map,
        'test_accuracy': float(test_accuracy),
        'image_size': 224
    }

    with open('model_info_final.json', 'w') as f:
        json.dump(class_info, f, indent=2)
    print("✅ Class info saved: model_info_final.json")

    import zipfile
    with zipfile.ZipFile('fish_disease_model_complete.zip', 'w') as zipf:
        zipf.write('fish_disease_model_final.h5')
        zipf.write('model_info_final.json')

    print("✅ Package created: fish_disease_model_complete.zip")

    print("\n⬇️ Downloading model...")
    files.download('fish_disease_model_complete.zip')

    return test_accuracy

In [ ]:
# %%
# 9. MAIN FUNCTION
def main():
    """Main training pipeline"""
    print("="*70)
    print("🐟 FISH DISEASE DETECTION - SIMPLIFIED TRAINING")
    print("="*70)

    try:
        dataset_path = upload_dataset()
        if not dataset_path:
            return

        print("\n🔄 Loading dataset...")
        result = load_dataset_from_folders(dataset_path)

        if result[0] is None:
            print("❌ Failed to load dataset!")
            return

        images, labels, class_names, reverse_label_map = result

        images = images.astype('float32') / 255.0

        print(f"\n📊 Dataset Info:")
        print(f"  Total images: {len(images)}")
        print(f"  Classes: {len(class_names)}")
        print(f"  Class names: {class_names}")

        unique, counts = np.unique(labels, return_counts=True)
        print(f"\n📈 Class Distribution:")
        for label_idx, count in zip(unique, counts):
            class_name = reverse_label_map[label_idx]
            percentage = (count / len(labels)) * 100
            print(f"  {class_name}: {count} images ({percentage:.1f}%)")

        print("\n✂️ Splitting data...")
        X_train, X_temp, y_train, y_temp = train_test_split(
            images, labels, test_size=0.3, random_state=42, stratify=labels
        )

        X_val, X_test, y_val, y_test = train_test_split(
            X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
        )

        print(f"  Training: {len(X_train)} images")
        print(f"  Validation: {len(X_val)} images")
        print(f"  Test: {len(X_test)} images")

        model = create_simple_model(len(class_names))

        epochs = int(input("\nEnter epochs (default 20): ") or "20")
        batch_size = int(input("Enter batch size (default 32): ") or "32")

        history = train_simple_model(
            model, X_train, y_train, X_val, y_val,
            epochs=epochs, batch_size=batch_size
        )

        test_accuracy = evaluate_and_visualize(model, X_test, y_test, class_names, history)

        save_final_model(model, class_names, reverse_label_map, test_accuracy)

        print("\n" + "="*70)
        print("🎉 TRAINING COMPLETE!")
        print("="*70)
        print(f"📊 Final Test Accuracy: {test_accuracy*100:.2f}%")
        print("📁 Model saved successfully!")
        print("\n✅ Ready for deployment!")
        print("="*70)

    except Exception as e:
        print(f"\n❌ Error: {e}")
        import traceback
        traceback.print_exc()

# %%
# 10. RUN
if __name__ == "__main__":
    np.random.seed(42)
    tf.random.set_seed(42)
    random.seed(42)
    main()